# Python Project Setup with uv

Before any of the modelling in this course, one practical problem has to be solved: making sure that the
code you write today still runs next month, and that it runs on your classmate's laptop as well as your
own.

That problem has a standard solution (a virtual environment plus a recorded dependency list) and a modern
tool that makes it painless: **uv**.

This notebook is short on theory and long on the things that actually go wrong. In particular it settles
one question that trips up almost everyone: uv offers two different workflows, they look
interchangeable, and they are not. Section 4 measures the difference.

This is `00`, and it is built around one idea: once an environment is set up, real code moves out of
notebook cells and into `.py` files that can be run the same way every time. The rest of this notebook is
uv material and the project you will build both final assignments in. Section 9, near the end, explains how the course builds on
it.

## How to use this notebook

**When.** This notebook is homework, not covered in class: work through it, and the practice project at
the end, **before the first lesson**. The first lesson starts with the theory of notebook `01`; time left
at the end goes to environment problems, and whatever is still open is solved at the start of the second
lesson of week 2, before notebook `04`. Come able to show three things:

1. `sys.executable` in a notebook cell and `uv run python -c "import sys; print(sys.executable)"` in a
   terminal, both pointing into a `.venv` folder (sections 6 and 7);
2. `uv run python main.py` printing the overview of your practice project;
3. notebook `01` running in the course repository, after `uv sync`.

Run `uv sync` at home, not on the school network just before the lesson: PyTorch is a large download.
Stuck? Section 8 lists the five errors behind most environment problems; bring the error message to the
first lesson.

**What you need.** A terminal and uv. Install it from [docs.astral.sh/uv](https://docs.astral.sh/uv/):
one command on any platform, no Python required beforehand.

**Where the work happens.** Most commands here are terminal commands, not notebook cells. Read them in the
notebook, type them in a terminal. The code cells that *do* run are checks: they tell you whether your
environment is what you think it is.

**Time.** About 1 hour for the notebook, plus the practice project.

**The one thing to take away.** Use `uv add`, not `uv pip install`. Section 4 shows what happens if you
forget, and the failure is silent until somebody else tries to run your project.

## Learning objectives

After this notebook, you can:

- explain what a virtual environment is and what problem it solves, with a concrete example;
- set up a project with `uv init`, `uv add` and `uv run`;
- say what each of `.venv`, `pyproject.toml`, `uv.lock` and `.python-version` is for;
- explain why `uv pip install` breaks reproducibility, and demonstrate it;
- check whether your running environment matches what your project declares;
- point a notebook at the right kernel, and recognise the symptom when it is not;
- diagnose the five errors that account for most environment problems;
- explain, in one sentence, why real code moves out of notebook cells into `.py` files.

## 1. The problem environments solve

Suppose you have two projects on one laptop. Project A was written last year against pandas 1.5, where
`df.append()` exists. Project B is this course, on pandas 2.x, where that method was removed.

Install packages globally and there is exactly one pandas on your machine. Whichever version you pick,
one of the two projects is broken, and the error you get (`AttributeError: 'DataFrame' object has no
attribute 'append'`) says nothing about versions at all.

A **virtual environment** is a private Python installation for one project: its own interpreter, its own
packages, its own versions. Project A gets pandas 1.5, project B gets pandas 2.2, and neither knows the
other exists.

That solves half the problem. The other half is **writing down** what your project needs, so that the
environment can be rebuilt: on another machine, next year, or after you delete it by accident. An
environment that only exists on your laptop is not reproducible; it is just a folder.

So two things are needed, and uv handles both:

| What | Where it lives | What it is for |
|---|---|---|
| The environment | `.venv/` | the actual installed packages, on your machine only |
| The declaration | `pyproject.toml` | what your project needs, in broad terms |
| The exact record | `uv.lock` | precisely which versions were resolved, for everyone |

The last two go into version control. `.venv/` never does: it is large, machine-specific, and rebuildable
in seconds from the other two.

## 2. uv, and the one decision

uv is a package and project manager: it creates environments, installs packages, manages Python versions
themselves, and runs your code in the right place. It is a single fast tool covering what used to take
`venv`, `pip`, `pip-tools` and `pyenv` together.

It offers **two workflows**, and this is the decision to get right at the start.

| | Project workflow | pip-compatible interface |
|---|---|---|
| Commands | `uv init`, `uv add`, `uv run`, `uv sync` | `uv venv`, `uv pip install` |
| Records in `pyproject.toml` | **yes** | **no** |
| Writes `uv.lock` | **yes** | **no** |
| Needs activation | no | yes |
| Reproducible | yes | only if you maintain a list by hand |

**Use the project workflow.** The pip interface exists so that uv can be dropped into an existing
pip-based project without changing anything, and as an escape hatch for one-off installs. For your own
work it costs you reproducibility in exchange for nothing.

The next two sections show the good path and then demonstrate exactly what the other one costs.

**Key terms.** *Virtual environment*: a private Python installation for one project. *Dependency*: a
package your project needs. *Reproducible*: someone else can rebuild your environment from your files.

## 3. The project workflow, in four commands

```bash
uv init my_project        # create the project scaffolding
cd my_project
uv add pandas             # add a dependency: installs it AND records it
uv run python main.py     # run in the project environment
```

That is the whole thing. A few notes on each.

**`uv init my_project`** creates a directory containing `pyproject.toml`, `main.py`, `README.md`,
`.python-version` and a `.gitignore`, the last of which already excludes `.venv/`, so you cannot commit
it by accident.

**`uv add pandas`** does three things at once: creates `.venv/` if it does not exist, installs pandas into
it, and writes the dependency into `pyproject.toml` and `uv.lock`. Three jobs, one command, and nothing
left implicit.

**`uv run python main.py`** runs your script inside the project environment. Note what you did **not**
have to do: activate anything. `uv run` finds the project's environment itself, and it syncs the
environment with your declared dependencies first, so it cannot run against a stale one.

**`uv sync`** rebuilds the environment to match `pyproject.toml` and `uv.lock` exactly. This is the
command your classmate runs after cloning your repository, and it is the moment of truth for whether your
project is reproducible.

### Activation, and why you can skip it

You will see this everywhere:

```bash
source .venv/bin/activate          # macOS, Linux
.\.venv\Scripts\Activate.ps1       # Windows PowerShell
.venv\Scripts\activate.bat         # Windows Command Prompt
```

Activation puts the environment's Python first on your `PATH`, so that plain `python` means the project's
python. It works fine and you may prefer it. But with `uv run` it is unnecessary, and skipping it removes
a whole category of mistakes, because you can no longer forget to activate, activate the wrong one, or
lose the activation when you open a second terminal tab.

On Windows, activation also runs into PowerShell's execution policy (see the troubleshooting section).
`uv run` sidesteps that too.

## 4. Why `uv pip install` is a trap

Both of these appear to work. Only one of them is reproducible, and the difference is invisible until
somebody else tries to run your project.

The commands below were run in a scratch project. Read the outputs rather than taking the claim on trust.

```bash
$ uv init demo && cd demo
$ uv add six
 + six==1.17.0
```

`pyproject.toml` afterwards:

```toml
dependencies = [
    "six>=1.17.0",
]
```

Now the other way, in the same project:

```bash
$ uv pip install toml
 + toml==0.10.2
```

`pyproject.toml` afterwards:

```toml
dependencies = [
    "six>=1.17.0",
]
```

**Unchanged.** `toml` is installed and importable (`uv run python -c "import toml"` works fine), but
nothing anywhere records that your project needs it. It is not in `pyproject.toml` and not in `uv.lock`.

Everything keeps working on your machine, which is exactly why the mistake survives. Then:

```bash
$ uv sync
Resolved 2 packages in 2ms
Uninstalled 1 package in 0.41ms
 - toml==0.10.2
```

`uv sync` rebuilds the environment from what the project *declares*, so it removes the package that was
never declared. Your code now fails with `ModuleNotFoundError: No module named 'toml'`, on your own
machine, after a command that was supposed to be harmless.

For a classmate cloning your repository, the same thing happens on the first run. Your project simply does
not work, and the missing package is not mentioned anywhere they can look.

> **The rule: `uv add` for anything your project needs.** Keep `uv pip install` for throwaway experiments
> in a throwaway environment.

If you have already done it, the fix is easy: run `uv add` for the packages that are missing from
`pyproject.toml`. Section 6 has a cell that finds them for you.

## 5. The four files

After `uv init` and a few `uv add`s, your project contains four things that matter. Three go into git and
one does not.

| File | Contents | In git? |
|---|---|---|
| `pyproject.toml` | project metadata and the dependencies you asked for, loosely pinned (`pandas>=2.0`) | **yes** |
| `uv.lock` | every package that was actually resolved, exact versions, including the dependencies of your dependencies | **yes** |
| `.python-version` | which Python version this project uses | **yes** |
| `.venv/` | the installed packages themselves, thousands of files, machine-specific | **no** |

The distinction between the first two is worth understanding. `pyproject.toml` says what you *want*:
"pandas, at least version 2.0". `uv.lock` records what you *got*: pandas 2.2.3, plus numpy 2.1.1, plus
python-dateutil, plus the sixty other packages that came along. Committing the lock file is what makes
"works on my machine" reproducible on someone else's.

This repository is itself a uv project, so you can look at a real example.

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for name in ["pyproject.toml", "uv.lock", ".python-version", ".venv"]:
    path = project_root / name
    if path.exists():
        size = f"{sum(1 for _ in path.open()) } lines" if path.is_file() else "a directory"
        print(f"{name:<18} present  ({size})")
    else:
        print(f"{name:<18} not present here")

In [ ]:
try:
    import tomllib                     # standard library from Python 3.11 onwards
except ModuleNotFoundError:
    import tomli as tomllib            # on Python 3.10: uv add tomli

with (project_root / "pyproject.toml").open("rb") as handle:
    configuration = tomllib.load(handle)

project = configuration["project"]
print(f"name           : {project['name']}")
print(f"python required: {project['requires-python']}")
print("dependencies   :")
for dependency in project["dependencies"]:
    print(f"  - {dependency}")

extras = project.get("optional-dependencies", {})
if extras:
    print("optional groups:", ", ".join(extras))
    print("  install one with:  uv sync --extra yolo")

Note the loose pins: `pandas>=2.0` rather than `pandas==2.2.3`. That is deliberate: the `pyproject.toml`
states a requirement, and `uv.lock` records the exact resolution. You get flexibility in the declaration
and precision in the record.

Note also the optional group. `uv add --optional yolo ultralytics` puts a dependency in a group that is
only installed when asked for, which is how this project keeps a large optional dependency out of everyone's
default install.

## 6. Checking that your environment is what you think it is

Three checks, in order of how often they catch something.

**Check one: which Python am I actually running?**

In [ ]:
import sys

print(sys.executable)
print()
if ".venv" in sys.executable:
    print("this interpreter is inside a .venv - good")
else:
    print("this interpreter is NOT in a project .venv")
    print("if you expected it to be, see section 7: your notebook kernel is probably wrong")

The path should contain `.venv` and should sit inside your project folder. If it points at something like
`/usr/bin/python3`, `C:\Python312\python.exe` or a conda directory, you are running against a different
Python than the one your project set up, which is the cause of most "but I installed it!" confusion.

**Check two: are the packages importable, and which versions?**

In [ ]:
from importlib.metadata import PackageNotFoundError, version

for package in ["pandas", "numpy", "matplotlib", "scikit-learn", "torch"]:
    try:
        print(f"{package:<14} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:<14} NOT INSTALLED in this environment")

Note `importlib.metadata.version` rather than `import pandas; pandas.__version__`. It reads the installed
package metadata, which works for packages whose import name differs from their distribution name:
`scikit-learn` is imported as `sklearn`, and `pandas.__version__` would not have told you about it at all.

**Check three: does what is installed match what the project declares?**

This is the check that catches the `uv pip install` trap from section 4.

In [ ]:
import re
from importlib.metadata import distributions

declared = {re.split(r"[><=\[;\s]", item)[0].lower().replace("_", "-")
            for item in project["dependencies"]}
installed = {distribution.metadata["Name"].lower().replace("_", "-")
             for distribution in distributions()
             if distribution.metadata["Name"]}

missing = sorted(declared - installed)
print("declared in pyproject.toml but NOT installed:", missing or "none")
if missing:
    print("  -> run: uv sync")
print()
print(f"({len(installed)} distributions installed in total, most of them dependencies of dependencies)")

The reverse direction (installed but not declared) is harder to check automatically, because most
installed packages are legitimately dependencies of your dependencies rather than things you asked for.
The reliable version of that check is simply `uv sync`: if your code stops working afterwards, something
was installed without being declared.

**Key terms.** *Interpreter*: the specific `python` executable running your code. *Distribution name*:
what a package is called on PyPI, which is not always its import name. *`uv sync`*: rebuild the
environment to match the project files exactly.

### Exercise 1: set up and verify

**A. Look and read off** (3 minutes). Run the three check cells above. Which interpreter are you on, and
are all five packages installed? If any check surprises you, that is the exercise doing its job.

**B. Do it yourself** (20 minutes). In a terminal, outside this repository:

1. Create a project with `uv init scratch_project` and look at every file it made.
2. `uv add pandas`, then open `pyproject.toml` and `uv.lock`. How many packages are in the lock file, and
   why is it so many more than what you asked for?
3. Run `uv run python -c "import sys; print(sys.executable)"`. Where is that interpreter?
4. Now delete the whole `.venv` folder and run the same command again. What happens, and how long does it
   take?

**C. Going further** (optional). In the same project, `uv pip install requests`, confirm that it imports,
then run `uv sync` and try again. Explain the result to yourself in one sentence before reading section 4
again.

In [ ]:
# This exercise is terminal work. Notes here, or in a markdown cell.

<details>
<summary>Worked answer: Exercise 1</summary>

**B.**

1. `pyproject.toml`, `main.py`, `README.md`, `.python-version`, `.gitignore` and a `.git` directory: uv
   initialises version control for you, and its `.gitignore` already excludes `.venv/`.
2. `pyproject.toml` lists one dependency; `uv.lock` contains pandas *and* everything pandas needs: numpy,
   python-dateutil, pytz, six and so on. That is the point of a lock file: your classmate needs the same
   versions of the indirect dependencies too, because those can break your code just as easily.
3. Inside `scratch_project/.venv/`. `uv run` found it without any activation.
4. It rebuilds the environment and runs, usually in a second or two. This is the whole promise of the
   approach: `.venv/` is disposable because `pyproject.toml` and `uv.lock` are not. It is also why
   committing `.venv/` to git is pointless as well as messy.

**C.** `requests` imports fine, and after `uv sync` it is gone: uninstalled, because nothing declares it.
One sentence: *installing without declaring gives you a working environment that nobody else can rebuild,
including future you.*

</details>

## 7. Notebooks, kernels and VS Code

A notebook does not run in "your project". It runs in a **kernel**, and the kernel is a specific Python
interpreter that you choose, or that got chosen for you, usually wrongly.

This is the single most common source of confusion for students in this course:

> You run `uv add pandas`, it installs successfully, you go back to your notebook, and
> `import pandas` raises `ModuleNotFoundError`.

Nothing is broken. The package went into your project's `.venv`, and the notebook is running on a
different Python. Check one in section 6 tells you which.

### Setting it up in VS Code

1. Open the **project folder** in VS Code, not a parent folder, not just the notebook file. The Python
   extension looks for `.venv` relative to the workspace root.
2. Make sure the environment exists: `uv sync` in a terminal.
3. Open the notebook, click the kernel selector in the top right, choose **Python Environments**, and pick
   the interpreter inside your project's `.venv`.
4. Run the `sys.executable` cell from section 6 to confirm.

If the environment does not appear in the list, reload the window (Command Palette →
*Developer: Reload Window*); the extension scans on startup.

### Jupyter Lab or the classic notebook

If you prefer Jupyter over VS Code, run it from inside the project so that it inherits the environment:

```bash
uv add --dev jupyterlab
uv run jupyter lab
```

The `--dev` flag puts it in a development group, keeping it out of your project's real dependencies:
your project needs pandas to run, but it does not need Jupyter.

### The kernel check, as a habit

Put this at the top of any notebook where you are unsure. It costs one line and saves an afternoon:

```python
import sys; print(sys.executable)
```

## 8. Troubleshooting

The five things that actually go wrong, in rough order of frequency.

| Symptom | Almost always | Fix |
|---|---|---|
| `ModuleNotFoundError` for something you just installed | notebook kernel points at a different Python | check `sys.executable`, then select the `.venv` kernel (section 7) |
| Code works in the terminal, fails in the notebook | same thing | same thing |
| `uv: command not found` | uv is not on your `PATH` | reopen the terminal after installing; on Windows a new PowerShell window |
| PowerShell refuses `Activate.ps1` | execution policy | `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`, or just use `uv run` and never activate |
| A package works for you and not for a classmate | installed with `uv pip install`, never declared | `uv add` it, commit `pyproject.toml` and `uv.lock` |
| `uv sync` removes a package you were using | the same cause, showing itself | as above |
| Import works, wrong version | an older environment, or a second install outside it | `uv sync`, then check `version()` from section 6 |
| `.venv` committed to git, repository enormous | `.gitignore` missing the entry | add `.venv/` to `.gitignore`, `git rm -r --cached .venv` |

Two habits that prevent most of these:

- **Start every debugging session with `sys.executable`.** Half of all environment problems are answered
  by that one line, and guessing wastes far more time than checking.
- **`uv sync` after every `git pull`.** Someone else may have added a dependency, and the lock file is how
  you find out.

### Exercise 2: diagnose

**A. Look and read off** (3 minutes). For each symptom, name the first thing you would check:
your classmate's project fails to run after you clone it; your notebook cannot find pandas; your script
uses pandas 1.5 when you installed 2.2.

**B. Do it yourself** (20 minutes). Reproduce two of these deliberately, in your scratch project. You
will recognise them far faster afterwards:

1. Make the notebook kernel wrong: start Jupyter or VS Code against a different interpreter and watch the
   import fail. Then fix it and confirm with `sys.executable`.
2. Install something with `uv pip install`, use it in a script, run `uv sync`, and watch the script break.
3. Delete `.venv` and rebuild it with `uv sync`. Time it.

**C. Going further** (optional). Write a three-line "getting started" section for your project's README:
the exact commands another person needs to run your code after cloning. Then test it by cloning your own
project into a different folder and following your own instructions literally.

In [ ]:
# Terminal work again. Notes here.

<details>
<summary>Worked answer: Exercise 2</summary>

**A.** For the clone that fails: whether `pyproject.toml` and `uv.lock` are actually in the repository,
then `uv sync`. For pandas not found in a notebook: `sys.executable`, every time. For the wrong version:
whether you are in the environment you think you are (again `sys.executable`), and then `uv sync`.

Notice that the same check answers two of the three. That is why it is the first thing to run.

**B.** Number 2 is the one worth doing properly. Watching a script break because of a command you ran two
days ago is a better teacher than any warning box, and it makes the `uv add` rule stick.

**C.** The point of the exercise is the test at the end. Nearly every README fails it the first time:
usually a missing `cd`, an unmentioned data folder, or an assumption that the reader already has uv. Three
lines that genuinely work are worth more than a page that nearly does:

```bash
git clone <url> && cd <project>
uv sync
uv run python main.py
```

</details>

## Practice project: an image-folder overview

Create a standalone project managed with uv, containing one script that reads a folder of images and
prints a clearly labelled overview. The point is the project setup as much as the script.

This is practice, not handed in on its own. But it is the project both final assignments are built in. If another person cannot run with `uv sync` and `uv run`, it is not finished. So set it up properly now.

### 1. Set up

```bash
uv init P2_computervision
cd P2_computervision
uv add pillow
```

Create a folder `data/images` with **one subfolder per class**: at least two classes and at least twenty
images in total. The folder name is the label: `data/images/cats/…`, `data/images/dogs/…`. Use your own
photographs or images you are allowed to share, and no recognizable people.

```text
P2_computervision/
    .venv/            <- not in git
    data/
        images/
            class_a/
            class_b/
    main.py
    pyproject.toml    <- in git
    uv.lock           <- in git
    README.md
    .gitignore
```

### 2. Write the overview

`main.py` looks at `data/images` and prints, each with a clear label:

1. the path of the folder;
2. the number of files, and the number of classes (subfolders);
3. the names of the classes, and the number of files in each;
4. the first 10 files, each with its class and its width × height in pixels.

Someone should be able to understand the output without reading your code. Leave the images unchanged.

Useful: `Path("data/images").iterdir()`, `folder.glob("*")`, `Image.open(path)`, `image.size`.

```bash
uv run python main.py
```

### 3. Extend it

Add:

- the colour mode of each image (`image.mode`: `RGB`, `L`, `RGBA`), counted per mode;
- the smallest, the median and the largest width and height;
- every file that cannot be opened as an image, with the error. Put a text file in one of your folders to
  test it.

Optional: accept the folder as a command-line argument, so the script works on any image folder.

If any of those outputs raises a question about your images (an unexpected mode, one image far smaller than
the rest), write the question down. That is notebook `04`, and you have just started it early.

### 4. Show that it is reproducible

This section is the actual subject of this notebook, so it carries the most weight.

- Check that `pillow` appears in `pyproject.toml`. **If it does not, you used `uv pip install`**. Fix it
  with `uv add pillow` (section 4).
- Check that `uv.lock` exists and is committed.
- Run `uv run python -c "import sys; print(sys.executable)"` and confirm the interpreter is in your
  project's `.venv`.
- Delete `.venv`, run `uv sync`, and run your script again. If it works, your project is reproducible. If
  it does not, you have found something worth fixing before you build on it.
- Keep `.venv/` out of git.

### 5. The README

Write, in your own words:

- the exact commands another person runs to set up and run your project;
- a short description of the images and where they came from;
- what `.venv`, `pyproject.toml` and `uv.lock` each do, and why you use `uv run`.

Then test the instructions by following them literally in a fresh folder.

### Check yourself

In git: `main.py`, `pyproject.toml`, `uv.lock`, `.gitignore` and `README.md`. Not `.venv/`, and not the
images if they are large or not yours to share.

Check the number of files, the classes and the sizes of the first ten images by hand. You are done when
the script runs with `uv run python main.py`, prints everything above, and a classmate can follow your
README on their own machine. The final assignments ask exactly that of their code.

## Typical mistakes

| Mistake | What you see | What to do |
|---|---|---|
| `uv pip install` instead of `uv add` | works for you, broken for everyone else | `uv add`; check `pyproject.toml` afterwards |
| Wrong notebook kernel | `ModuleNotFoundError` for something you just installed | `sys.executable`, then select the `.venv` kernel |
| `.venv/` committed to git | a repository of thousands of files | `.gitignore` it; `git rm -r --cached .venv` |
| `uv.lock` *not* committed | a classmate gets different versions | commit it; that is the whole point |
| Opening the wrong folder in VS Code | the environment is never found | open the project root |
| Forgetting `uv sync` after `git pull` | a dependency somebody else added is missing | make it a habit |
| Installing globally "just this once" | version conflicts across projects | always inside a project |
| A README that was never tested | nobody can run your project | follow your own instructions in a fresh folder |

## Glossary

| Term | In one line |
|---|---|
| Virtual environment | a private Python installation for one project |
| `.venv/` | where that environment lives; never in git |
| `pyproject.toml` | project metadata and the dependencies you asked for |
| `uv.lock` | the exact versions that were resolved, including indirect ones |
| `.python-version` | which Python version this project uses |
| Dependency | a package your project needs |
| Indirect dependency | a package your dependencies need |
| `uv add` | install **and** record a dependency |
| `uv sync` | rebuild the environment to match the project files |
| `uv run` | run a command in the project environment, no activation needed |
| Activation | putting an environment's Python first on your `PATH` |
| Kernel | the interpreter a notebook runs in |
| Reproducible | someone else can rebuild your environment from your files |

## Knowledge check

1. Why is a virtual environment not enough on its own to make a project reproducible?
2. What is the difference between `pyproject.toml` and `uv.lock`?
3. Which of the four project files does *not* go into version control, and why not?
4. What exactly does `uv add pandas` do that `uv pip install pandas` does not?
5. You installed a package and your notebook cannot import it. What do you check first?
6. Why does `uv run` mean you never have to activate anything?
7. Your classmate clones your repository and the code fails on a missing package. What went wrong?
8. Why does `uv.lock` contain far more packages than you ever asked for?
9. You deleted `.venv` by accident. How much work is that?

<details>
<summary>Suggested answers</summary>

1. Because the environment only exists on your machine. Reproducibility needs a *written record* of what
   is in it, which is what `pyproject.toml` and `uv.lock` provide.
2. `pyproject.toml` says what you want, loosely (`pandas>=2.0`). `uv.lock` records what you got, exactly
   (pandas 2.2.3 plus every indirect dependency). The first is a requirement, the second is a receipt.
3. `.venv/`: it is large, machine-specific, and rebuildable in seconds from the other three. Committing it
   adds thousands of files and helps nobody.
4. It writes the dependency into `pyproject.toml` and `uv.lock`. Both commands install the package; only
   `uv add` records it, and the difference is invisible until `uv sync` or another machine.
5. `sys.executable`. If it is not inside your project's `.venv`, the notebook kernel is wrong and nothing
   else you try will help.
6. Because `uv run` resolves the project environment itself before running the command, and syncs it with
   your declared dependencies first. Activation only changes your `PATH`, which `uv run` does not need.
7. A package was installed without being declared, almost certainly `uv pip install`. Or `uv.lock` was not
   committed. Both are section 4.
8. Because your dependencies have dependencies. pandas needs numpy, python-dateutil and pytz, each of which
   needs more. Your classmate needs the same versions of those too, because an indirect dependency can
   break your code just as easily as a direct one.
9. A couple of seconds: `uv sync`. That is the payoff for keeping the declaration and the lock file in git,
   and it is why `.venv/` being disposable is a feature rather than a risk.

</details>

## Recap

- A virtual environment isolates one project's packages. A written record makes it reproducible. You need
  both.
- Four files: `pyproject.toml`, `uv.lock` and `.python-version` go into git; `.venv/` does not.
- `uv init`, `uv add`, `uv run`, `uv sync`. That is the whole workflow, and `uv run` means you never have
  to activate anything.
- `uv add` records a dependency; `uv pip install` does not. The difference is silent until `uv sync`
  removes the package or a classmate cannot run your code.
- A notebook runs in a kernel, and the kernel is where "I installed it but it is not there" comes from.
  `sys.executable` answers it.
- Test your README by following it literally in a fresh folder.

